# Pipeline Adaptive Thresholds Notebook

This notebook is a notebook-first extension for the current pipeline.

It gives you four direct-run sections:

1. Load historical pipeline outputs
2. Compute rolling daily thresholds instead of fixed `0.20 / 0.30`
3. Try an RL-style threshold-pair bandit
4. Browse checkpoints and preview resume configs

It uses helper code from `adaptive_trade_extensions.py` and does not modify your existing pipeline notebooks.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from adaptive_trade_extensions import (
    RollingThresholdConfig,
    ThresholdPairBandit,
    ThresholdPairBanditConfig,
    NewsShockGuard,
    NewsShockConfig,
    build_proxy_reward_frame,
    build_equity_proxy_reward_frame,
    make_threshold_grid,
    select_moving_daily_thresholds,
    select_oracle_thresholds_from_daily_rewards,
    threshold_confusion_score,
    build_threshold_bandit_reward,
    make_daily_threshold_state,
)

ROOT = Path.cwd()
OUTPUT_DIRS = sorted([p for p in ROOT.glob('output*') if p.is_dir()])
CHECKPOINTS = sorted(ROOT.glob('checkpoints/**/*.joblib'))

print(f'Workspace: {ROOT}')
print(f'Output directories found: {len(OUTPUT_DIRS)}')
print(f'Checkpoint files found: {len(CHECKPOINTS)}')

In [ ]:
# ------------------------------------------------------------
# Pick one output folder that has daily_log.csv
# ------------------------------------------------------------

daily_log_candidates = [p for p in ROOT.glob('output*/daily_log.csv')]
daily_log_candidates = sorted(daily_log_candidates, key=lambda p: p.stat().st_mtime, reverse=True)

for i, p in enumerate(daily_log_candidates[:20]):
    print(f'[{i}] {p}')

# Change this index to the run you want to analyze.
selected_idx = 0
daily_log_path = daily_log_candidates[selected_idx]
daily_log_path

In [ ]:
# ------------------------------------------------------------
# Load daily_log and create a history frame for threshold tuning
# ------------------------------------------------------------

daily_log_df = pd.read_csv(daily_log_path)
if 'date' in daily_log_df.columns:
    daily_log_df['date'] = pd.to_datetime(daily_log_df['date'])

daily_log_df.head()

In [ ]:
# ------------------------------------------------------------
# Build the reward frame used by adaptive threshold logic
#
# Preferred inputs:
#   sibling daily_bandit_log.csv from the RL / exact-walkforward pipeline
#   because that file already contains 5m-based per-day rewards for:
#   reward_force_buy / reward_free / reward_force_sell
#
# Fallback:
#   if daily_bandit_log.csv does not exist, use next_day_return if available
#   if not, infer a weak proxy from equity
# ------------------------------------------------------------

bandit_log_path = daily_log_path.with_name('daily_bandit_log.csv')

if bandit_log_path.exists():
    print(f'Using sibling daily_bandit_log.csv: {bandit_log_path}')
    hist_df = pd.read_csv(bandit_log_path)
    if 'date' in hist_df.columns:
        hist_df['date'] = pd.to_datetime(hist_df['date'])
else:
    hist_df = daily_log_df.copy()

if {'reward_force_buy', 'reward_free', 'reward_force_sell'}.issubset(hist_df.columns):
    print('Using explicit reward columns from daily_log.csv')
elif 'next_day_return' in hist_df.columns:
    print('Using proxy reward frame from next_day_return')
    hist_df = build_proxy_reward_frame(hist_df, next_day_ret_col='next_day_return')
elif 'equity' in hist_df.columns:
    print('Using proxy reward frame inferred from equity')
    hist_df = build_equity_proxy_reward_frame(hist_df, equity_col='equity')
else:
    raise ValueError(
        'Need either reward_force_buy/reward_free/reward_force_sell, next_day_return, or equity in daily_log.csv.'
    )

required_cols = ['p_day', 'reward_force_buy', 'reward_free', 'reward_force_sell', 'best_action_ex_post']
display(hist_df[[c for c in required_cols if c in hist_df.columns]].head())

## Rolling Daily Thresholds

This replaces the fixed daily cutoffs like `p_buy_level=0.20` and `p_sell_level=0.30` with a rolling optimizer over recent daily history.

In [ ]:
# ------------------------------------------------------------
# Choose the current p_day you want to classify.
# By default we use the latest p_day in the selected run.
# ------------------------------------------------------------

current_p_day = float(hist_df['p_day'].dropna().iloc[-1])
current_p_day

In [ ]:
cfg = RollingThresholdConfig(
    lookback_days=60,
    buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
    sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
    min_gap=0.02,
    min_obs=20,
    switch_penalty=0.0,
)

adaptive = select_oracle_thresholds_from_daily_rewards(
    history_df=hist_df,
    current_p_day=current_p_day,
    config=cfg,
    prev_buy_level=0.20,
    prev_sell_level=0.30,
    objective='reward',
)

adaptive

In [ ]:
# ------------------------------------------------------------
# Roll the optimizer through time and inspect how thresholds move
# ------------------------------------------------------------

rolling_rows = []
for i in range(len(hist_df)):
    sub = hist_df.iloc[:i+1].copy()
    p_now = float(sub['p_day'].dropna().iloc[-1]) if sub['p_day'].notna().any() else np.nan
    out = select_oracle_thresholds_from_daily_rewards(
        history_df=sub,
        current_p_day=p_now,
        config=cfg,
        prev_buy_level=0.20,
        prev_sell_level=0.30,
        objective='reward',
    )
    rolling_rows.append({
        'date': sub['date'].iloc[-1] if 'date' in sub.columns else i,
        'p_day': p_now,
        'buy_level': out.buy_level,
        'sell_level': out.sell_level,
        'gate': out.gate,
        'score': out.score,
        'window_size': out.window_size,
    })

rolling_df = pd.DataFrame(rolling_rows)
rolling_df.tail(20)

In [ ]:
# ------------------------------------------------------------
# Optional: optimize for action-classification accuracy instead
# This measures whether the threshold regions agree with the ex-post best action.
# ------------------------------------------------------------

adaptive_acc = select_oracle_thresholds_from_daily_rewards(
    history_df=hist_df,
    current_p_day=current_p_day,
    config=cfg,
    prev_buy_level=0.20,
    prev_sell_level=0.30,
    objective='accuracy',
)

conf = threshold_confusion_score(hist_df.tail(cfg.lookback_days), adaptive_acc.buy_level, adaptive_acc.sell_level)
adaptive_acc, conf['accuracy']

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
plt.plot(rolling_df['date'], rolling_df['p_day'], label='p_day', linewidth=1.5)
plt.plot(rolling_df['date'], rolling_df['buy_level'], label='adaptive buy level', linestyle='--')
plt.plot(rolling_df['date'], rolling_df['sell_level'], label='adaptive sell level', linestyle='--')
plt.legend()
plt.title('Rolling Daily Thresholds vs p_day')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## RL-Style Threshold Pair Bandit

This keeps your daily probability model, but lets a contextual bandit choose the `(buy_level, sell_level)` pair instead of fixing it.

In [ ]:
bandit_cfg = ThresholdPairBanditConfig(
    threshold_pairs=[
        (0.10, 0.25),
        (0.15, 0.30),
        (0.20, 0.30),
        (0.20, 0.35),
        (0.25, 0.40),
        (0.25, 0.50),
    ],
    alpha=0.50,
    l2=1.0,
)

bandit = ThresholdPairBandit(n_features=6, config=bandit_cfg)
bandit.threshold_pairs

In [ ]:
# ------------------------------------------------------------
# Offline pass: learn threshold-pair choices from the historical frame
#
# This is a simple direct-run demo. It uses p_day plus a few state fields.
# If your daily_log has better daily-state features, plug them in here.
# ------------------------------------------------------------

bandit_rows = []
current_pos = 0
cum_equity = 1.0
equity_peak = 1.0

for _, row in hist_df.iterrows():
    p_day = float(row['p_day']) if pd.notna(row['p_day']) else 0.5
    drawdown_rel = max(0.0, (equity_peak - cum_equity) / max(equity_peak, 1e-12))

    x = make_daily_threshold_state(
        p_day=p_day,
        dp_min=0.0,
        dp_max=0.0,
        realized_vol_20=0.0,
        drawdown_rel=drawdown_rel,
        current_pos=current_pos,
    )

    decision = bandit.decide_gate(x=x, p_day=p_day)
    reward = build_threshold_bandit_reward(
        gate=decision['gate'],
        force_buy_reward=float(row['reward_force_buy']),
        free_reward=float(row['reward_free']),
        force_sell_reward=float(row['reward_force_sell']),
    )

    bandit.update(decision['action_idx'], x, reward)

    cum_equity *= (1.0 + reward)
    equity_peak = max(equity_peak, cum_equity)
    current_pos = 1 if decision['gate'] == 'FORCE_BUY' else 0 if decision['gate'] == 'FORCE_SELL' else current_pos

    bandit_rows.append({
        'date': row['date'] if 'date' in row else None,
        'p_day': p_day,
        'buy_level': decision['buy_level'],
        'sell_level': decision['sell_level'],
        'gate': decision['gate'],
        'reward': reward,
        'cum_equity': cum_equity,
    })

bandit_df = pd.DataFrame(bandit_rows)
bandit_df.tail(20)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(bandit_df['date'], bandit_df['cum_equity'], label='threshold-pair bandit equity')
plt.legend()
plt.title('Offline Threshold-Pair Bandit Equity Proxy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

bandit_df[['buy_level', 'sell_level', 'gate']].tail(20)

## News Shock Guard

This is a separate override layer for high-impact market news. It is intentionally separate from the normal daily and 5-minute models.

In [ ]:
guard = NewsShockGuard(
    NewsShockConfig(
        high_impact_threshold=0.85,
        directional_threshold=0.40,
        halt_minutes=180,
    )
)

example_event = {
    'headline': 'Unexpected major policy speech shocks rate expectations',
    'impact_score': 0.92,
    'direction_score': -0.65,
    'confidence': 0.88,
}

guard.decide(example_event)

## Checkpoint Browser and Resume Preview

This section is notebook-style so you can inspect your checkpoint inventory and assemble resume calls directly.

In [ ]:
checkpoint_rows = []
for p in CHECKPOINTS:
    st = p.stat()
    checkpoint_rows.append({
        'path': str(p.relative_to(ROOT)),
        'modified': pd.to_datetime(st.st_mtime, unit='s'),
        'size_mb': round(st.st_size / (1024 * 1024), 2),
    })

checkpoint_df = pd.DataFrame(checkpoint_rows).sort_values('modified', ascending=False)
checkpoint_df.head(30)

In [ ]:
# ------------------------------------------------------------
# Pick a checkpoint and create a resume call preview
# ------------------------------------------------------------

checkpoint_idx = 0
selected_checkpoint = checkpoint_df.iloc[checkpoint_idx]['path']

resume_preview = f'''res = resume_from_checkpoint(
    checkpoint_path="{selected_checkpoint}",
    daily_csv_path="DataAPI/data/SPY_DAY.csv",
    k5m_csv_path="DataAPI/data/SPY_5M.csv",
    end_time="2026-12-31",
    output_dir="output_resumed_from_notebook",
    verbose=True,
    plot_from_checkpoint=True,
)'''

print(resume_preview)

In [ ]:
# ------------------------------------------------------------
# Optional: save threshold outputs for later comparison
# ------------------------------------------------------------

save_dir = ROOT / 'output' / 'adaptive_threshold_notebook'
save_dir.mkdir(parents=True, exist_ok=True)

rolling_df.to_csv(save_dir / 'rolling_thresholds.csv', index=False)
bandit_df.to_csv(save_dir / 'threshold_pair_bandit.csv', index=False)

summary = {
    'selected_daily_log': str(daily_log_path),
    'latest_adaptive_buy_level': float(adaptive.buy_level),
    'latest_adaptive_sell_level': float(adaptive.sell_level),
    'latest_adaptive_gate': adaptive.gate,
}

with open(save_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

summary